# Thesis figures: anatomical views and VinDr-CXR cohorts / bounding boxes

Creates two publication-ready figures **from real files**:

1. **Five-panel anatomical-view construction:** original CXR, lung mask, heart mask, lung-masked CXR, heart-masked CXR.
2. **VinDr-CXR prevalence and annotated examples:** verified image-level positive counts in frozen validation/test CSVs, plus two real positive images with radiologist bounding boxes.

**Before running:** edit the configuration cell. This notebook does not download datasets, generate masks, assume label prevalence, or synthesize boxes. It raises actionable errors when required inputs are missing. Exports **PDF, SVG and 300-dpi PNG**, as well as computed-count CSVs. Compatible with your ALBEF `Image.composite(image, black, mask)` view creation.

Install if necessary: `pip install numpy pandas matplotlib pillow`.


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import patches
from matplotlib.ticker import MaxNLocator
from PIL import Image, ImageOps
from IPython.display import display

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 9,
    "axes.titlesize": 10, "axes.labelsize": 9,
    "xtick.labelsize": 8, "ytick.labelsize": 8,
    "pdf.fonttype": 42, "ps.fonttype": 42,
    "savefig.facecolor": "white", "figure.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
})
BLUE = "#346D98"
TEAL = "#438679"
ORANGE = "#C78448"
TEXT = "#253243"
LIGHT = "#E6EAF0"


## Configuration

Set paths to your existing images, **lung segmentation masks**, **heart segmentation masks**, frozen validation/test *image-level* label manifests, and radiologist annotation CSV. If the CSV schemas differ from defaults, adjust the column settings. Image IDs should be preserved exactly as strings.

Mask layout can be flat or prefix-sharded (`root/<first two ID characters>/<ID>.png`). The image folder can also be flat or prefix-sharded. **Masks are assumed to be grayscale alpha masks**: 0 means hidden, 255 means retained; intermediary values are allowed. This matches your evaluation script.

The VinDr annotated examples may require access to full-resolution images rather than the 256-pixel evaluation copy. The notebook supports normalized box coordinates, or absolute coordinates when the source annotation coordinate dimensions are supplied.

In [ ]:
# ========================= EDIT THESE PATHS =========================
ORIGINAL_CXR_DIR = Path("/path/to/original_cxrs")
LUNG_MASK_DIR    = Path("/path/to/lung_segmentation_masks")
HEART_MASK_DIR   = Path("/path/to/heart_segmentation_masks")

# Choose a real image with both matching masks; set to None for first matching ID.
ANATOMY_IMAGE_ID = None  # e.g. "abc123..." (without .png)

# Frozen VinDr image-level label manifests (one row per radiograph).
VINDR_VAL_LABELS_CSV  = Path("/path/to/frozen_vindr_val_labels.csv")
VINDR_TEST_LABELS_CSV = Path("/path/to/frozen_vindr_test_labels.csv")
VAL_IMAGE_ID_COLUMN = None  # None: use first CSV column
TEST_IMAGE_ID_COLUMN = None

# VinDr bounding box annotations. Typically columns:
# image_id, class_name, x_min, y_min, x_max, y_max
VINDR_BBOX_CSV = Path("/path/to/vindr_bbox_annotations.csv")
ANNOTATED_CXR_DIR = Path("/path/to/vindr_annotated_images")
BBOX_ID_COL = "image_id"
BBOX_LABEL_COL = "class_name"
BBOX_COLS = ("x_min", "y_min", "x_max", "y_max")
# 'pixels' for absolute x/y, 'normalized' for values in 0..1.
BBOX_UNITS = "pixels"
# If annotation dimensions differ from the displayed CXR resolution,
# specify the annotation reference canvas below, e.g. (3000, 3000).
# Leave None only when image pixels equal the annotation coordinate system.
BBOX_REFERENCE_SIZE = None  # (width,height) for all examples, if known
# Optional per-row original coordinate dimensions in annotation CSV:
BBOX_REFERENCE_WIDTH_COL = None   # e.g. "image_width"
BBOX_REFERENCE_HEIGHT_COL = None  # e.g. "image_height"

# Two target findings: must exist in BOTH evaluation label CSVs.
TARGET_LABELS = ("Cardiomegaly", "Pleural effusion")
# Explicitly choose representative positive image IDs, if desired.
EXAMPLE_IMAGE_IDS = {"Cardiomegaly": None, "Pleural effusion": None}
# If None, select first eligible test-set image with an annotation for the target.

EXPECTED_VAL_IMAGES = 2000
EXPECTED_TEST_IMAGES = 3000
OUTPUT_DIR = Path("thesis_publication_figures")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Set True to stop on unexpected cohort sizes or overlapping image IDs.
STRICT_COHORT_CHECKS = True


## Utilities: matching real images and masks, exports

In [ ]:
IMAGE_EXTENSIONS = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".webp")

def require_file(p, kind="file"):
    p = Path(p)
    if not p.is_file():
        raise FileNotFoundError(f"Missing {kind}: {p}. Edit the configuration cell.")
    return p

def require_directory(p):
    p = Path(p)
    if not p.is_dir():
        raise FileNotFoundError(f"Missing directory: {p}. Edit the configuration cell.")
    return p

def locate_image(root, image_id):
    root = require_directory(root)
    image_id = str(image_id)
    candidates = []
    for ext in IMAGE_EXTENSIONS:
        candidates += [root / (image_id + ext), root / image_id[:2] / (image_id + ext)]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    return None

def list_ids(root):
    root = require_directory(root)
    return {p.stem for p in root.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS}

def select_anatomy_id():
    if ANATOMY_IMAGE_ID is not None:
        image_id = str(ANATOMY_IMAGE_ID)
        for root, label in ((ORIGINAL_CXR_DIR, "original CXR"),
                            (LUNG_MASK_DIR, "lung mask"),
                            (HEART_MASK_DIR, "heart mask")):
            if locate_image(root, image_id) is None:
                raise FileNotFoundError(f"{label} missing for {image_id} inside {root}")
        return image_id
    ids = list_ids(ORIGINAL_CXR_DIR) & list_ids(LUNG_MASK_DIR) & list_ids(HEART_MASK_DIR)
    if not ids:
        raise RuntimeError("No shared image ID across the original CXR, lung mask, and heart mask directories")
    return sorted(ids)[0]

def load_grayscale_image(path):
    with Image.open(require_file(path)) as im:
        return ImageOps.exif_transpose(im).convert("L")

def export_figure(fig, basename):
    files = []
    for ext in ("pdf", "svg", "png"):
        out = OUTPUT_DIR / f"{basename}.{ext}"
        fig.savefig(out, dpi=300, bbox_inches="tight", pad_inches=0.08)
        files.append(out)
    print("Exported:", *map(str, files), sep="\n  ")
    return files

def apply_image_style(ax, title):
    ax.set_title(title, loc="center", color=TEXT, fontweight="semibold", pad=7)
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)


## Figure 1 — Anatomical-view construction (five panels)

Displays grayscale masks as standalone images and applies the *original* masks (without thresholding) to the CXR using PIL's compositing operation. Image/mask dimensions must match exactly; no silent resizing or misalignment.

In [ ]:
image_id = select_anatomy_id()
image_path = locate_image(ORIGINAL_CXR_DIR, image_id)
lung_path  = locate_image(LUNG_MASK_DIR, image_id)
heart_path = locate_image(HEART_MASK_DIR, image_id)

cxr = load_grayscale_image(image_path)
lung_mask = load_grayscale_image(lung_path)
heart_mask = load_grayscale_image(heart_path)
if cxr.size != lung_mask.size or cxr.size != heart_mask.size:
    raise ValueError(f"Misaligned sizes: CXR={cxr.size}, lung={lung_mask.size}, heart={heart_mask.size}; "
                     "fix mask alignment upstream; notebook will not silently resize them")

background = Image.new("L", cxr.size, color=0)
lung_masked = Image.composite(cxr, background, lung_mask)
heart_masked = Image.composite(cxr, background, heart_mask)

panels = [
    (cxr, "(a) Original CXR"),
    (lung_mask, "(b) Lung mask"),
    (heart_mask, "(c) Heart mask"),
    (lung_masked, "(d) Lung-masked CXR"),
    (heart_masked, "(e) Heart-masked CXR"),
]
fig, axes = plt.subplots(1, 5, figsize=(14.5, 3.15), layout="constrained")
for ax, (im, label) in zip(axes, panels):
    ax.imshow(np.asarray(im), cmap="gray", vmin=0, vmax=255, interpolation="none")
    apply_image_style(ax, label)
export_figure(fig, "anatomical_view_construction_five_panel")
plt.show()
print(f"Image ID: {image_id} | Original: {image_path} | Lung: {lung_path} | Heart: {heart_path}")


## Figure 2 — VinDr-CXR target-label prevalence and genuine bounding-box examples

**Left:** the number of positive images for Cardiomegaly and Pleural Effusion in each *fixed* cohort, extracted from the respective CSVs. **Right:** one genuine radiograph with annotated boxes per target finding.

- The unit of prevalence is **unique images**, not number of boxes or number of radiologist rows.
- Accepted values for target label columns: `0`, `1`, and optionally `-1` (uncertain/unlabeled). `-1` is not counted as positive; the CSV audit prints its count. Any other values cause an error.
- To avoid accidental leakage, the notebook verifies IDs are unique within each cohort and non-overlapping between val and test. It checks 2,000/3,000 as configured.
- Annotation boxes must come from the provided bbox CSV. Boxes from multiple radiologists may overlap and are retained; **no synthetic boxes** are added.
- For annotation dimensions different from the image dimensions, set `BBOX_REFERENCE_SIZE` or source width/height columns. Otherwise coordinates are assumed to refer to the displayed image pixel grid; visual verification is required.

In [ ]:
def read_frozen_labels(csv_path, image_id_column=None, cohort="cohort"):
    df = pd.read_csv(require_file(csv_path, cohort + " CSV"), dtype={image_id_column: str} if image_id_column else None)
    id_col = image_id_column or df.columns[0]
    if id_col not in df:
        raise KeyError(f"Missing ID column {id_col!r} in {csv_path}")
    df[id_col] = df[id_col].astype(str).str.strip()
    if df[id_col].isna().any() or df[id_col].eq("").any():
        raise ValueError(f"Missing/empty image IDs in {csv_path}")
    duplicates = df[id_col].duplicated().sum()
    if duplicates:
        raise ValueError(f"{cohort} has {duplicates} duplicate image ID rows. Resolve before computing counts.")
    missing = [lab for lab in TARGET_LABELS if lab not in df.columns]
    if missing:
        raise KeyError(f"Missing label column(s) {missing} in {csv_path}; available columns: {list(df.columns)}")
    return df, id_col

val_df, val_id_col = read_frozen_labels(VINDR_VAL_LABELS_CSV, VAL_IMAGE_ID_COLUMN, "validation")
test_df, test_id_col = read_frozen_labels(VINDR_TEST_LABELS_CSV, TEST_IMAGE_ID_COLUMN, "test")

assert_counts = {"validation": (len(val_df), EXPECTED_VAL_IMAGES), "test": (len(test_df), EXPECTED_TEST_IMAGES)}
for name, (observed, expected) in assert_counts.items():
    if observed != expected:
        msg = f"{name}: observed {observed} images, expected {expected}"
        if STRICT_COHORT_CHECKS: raise ValueError(msg)
        print("WARNING:", msg)

overlap = set(val_df[val_id_col]) & set(test_df[test_id_col])
if overlap:
    msg = f"Validation/test leakage: {len(overlap)} overlapping image IDs, e.g. {list(sorted(overlap))[:3]}"
    if STRICT_COHORT_CHECKS: raise ValueError(msg)
    print("WARNING:", msg)

rows = []
for lab in TARGET_LABELS:
    for name, df in (("Validation", val_df), ("Test", test_df)):
        values = pd.to_numeric(df[lab], errors="raise")
        if not values.isin([-1, 0, 1]).all():
            raise ValueError(f"Unexpected {lab} labels in {name}: {values[~values.isin([-1,0,1])].unique()}")
        rows.append({"Finding": lab, "Cohort": name, "Positive images": int((values == 1).sum()),
                     "Total images": len(df), "Unknown (-1)": int((values == -1).sum()),
                     "Prevalence (%)": 100 * (values == 1).sum()/len(df)})
counts_df = pd.DataFrame(rows)
display(counts_df)
count_csv = OUTPUT_DIR / "vindr_fixed_cohort_verified_counts.csv"
counts_df.to_csv(count_csv, index=False)
print("Saved verified counts:", count_csv)


### Radiologist annotations: verify schema and choose genuine positive cases

The notebook requires bounding boxes in the supplied CSV and test-cohort label positivity for the selected representative image. For reproducibility you can set `EXAMPLE_IMAGE_IDS` explicitly instead of relying on the first eligible image.

In [ ]:
bbox_df = pd.read_csv(require_file(VINDR_BBOX_CSV, "VinDr bounding box CSV"), dtype={BBOX_ID_COL: str})
required_cols = [BBOX_ID_COL, BBOX_LABEL_COL, *BBOX_COLS]
missing = [col for col in required_cols if col not in bbox_df.columns]
if missing:
    raise KeyError(f"Missing bbox CSV columns: {missing}. Available: {list(bbox_df.columns)}")
bbox_df[BBOX_ID_COL] = bbox_df[BBOX_ID_COL].astype(str).str.strip()
for col in BBOX_COLS:
    bbox_df[col] = pd.to_numeric(bbox_df[col], errors="coerce")
bbox_df = bbox_df.dropna(subset=list(BBOX_COLS))
print("Annotation labels in CSV (first 35):", sorted(bbox_df[BBOX_LABEL_COL].dropna().astype(str).unique())[:35])

def get_example(lab):
    target = bbox_df[bbox_df[BBOX_LABEL_COL].astype(str).str.casefold() == lab.casefold()]
    if target.empty:
        raise ValueError(f"No genuine bounding boxes for {lab!r} in {VINDR_BBOX_CSV}")
    pos_test = set(test_df.loc[pd.to_numeric(test_df[lab]) == 1, test_id_col].astype(str))
    eligible = sorted(set(target[BBOX_ID_COL].astype(str)) & pos_test)
    if not eligible:
        raise ValueError(f"No {lab} annotated image appears positive in the test manifest. "
                         "Check whether bbox annotations correspond to the same frozen evaluation split.")
    manual_id = EXAMPLE_IMAGE_IDS.get(lab)
    if manual_id is not None:
        if str(manual_id) not in eligible:
            raise ValueError(f"Specified example {manual_id} is not a positive, annotated test image for {lab}")
        candidates = [str(manual_id)]
    else:
        candidates = eligible
    for image_id in candidates:
        path = locate_image(ANNOTATED_CXR_DIR, image_id)
        if path is not None:
            return image_id, path, target.loc[target[BBOX_ID_COL] == image_id].copy()
    raise FileNotFoundError(f"No image file found for {lab} annotated candidates under {ANNOTATED_CXR_DIR}")

examples = {lab: get_example(lab) for lab in TARGET_LABELS}
for lab, (img_id, path, boxes) in examples.items():
    print(f"{lab}: image_id={img_id}, source={path}, boxes={len(boxes)}")


### Generate combined two-part publication figure

Three aligned sections: **left** target-label prevalence (image counts), **right** two authentic annotated CXR cases. Scale boxes only when coordinate metadata explicitly supplies a reference canvas. If boxes fall outside a displayed image, halt rather than drawing misleading annotations.

In [ ]:
def boxes_for_display(boxes, image_size):
    W, H = image_size
    result = []
    for _, row in boxes.iterrows():
        x1, y1, x2, y2 = (float(row[col]) for col in BBOX_COLS)
        if BBOX_UNITS == "normalized":
            if min(x1,y1,x2,y2) < 0 or max(x1,y1,x2,y2) > 1:
                raise ValueError("Normalized boxes must have coordinates in [0,1]")
            sx, sy = W, H
        elif BBOX_UNITS == "pixels":
            if BBOX_REFERENCE_WIDTH_COL and BBOX_REFERENCE_HEIGHT_COL:
                refw = float(row[BBOX_REFERENCE_WIDTH_COL]); refh = float(row[BBOX_REFERENCE_HEIGHT_COL])
            elif BBOX_REFERENCE_SIZE is not None:
                refw, refh = BBOX_REFERENCE_SIZE
            else:
                refw, refh = W, H
            if refw <= 0 or refh <= 0: raise ValueError("Invalid annotation reference size")
            sx, sy = W/refw, H/refh
        else:
            raise ValueError("BBOX_UNITS must be 'pixels' or 'normalized'")
        coords = (x1*sx, y1*sy, x2*sx, y2*sy)
        a,b,c,d = coords
        if not (0 <= a < c <= W and 0 <= b < d <= H):
            raise ValueError(f"Invalid/out-of-image box {coords} for displayed image {image_size}. "
                             "Verify annotation reference dimensions and image preprocessing.")
        result.append(coords)
    return result

fig = plt.figure(figsize=(12.6, 4.2), layout="constrained")
gs = fig.add_gridspec(1, 3, width_ratios=[1.6, 1.0, 1.0], wspace=0.09)
ax_bar = fig.add_subplot(gs[0,0])
y = np.arange(len(TARGET_LABELS))
h = 0.28
for i, (name, colour, dy) in enumerate((("Validation", TEAL, -h/2), ("Test", BLUE, h/2))):
    data = counts_df.set_index(["Finding", "Cohort"])
    values = [int(data.loc[(lab,name), "Positive images"]) for lab in TARGET_LABELS]
    bars = ax_bar.barh(y+dy, values, height=h, color=colour, label=name, zorder=3)
    for bar, value in zip(bars, values):
        ax_bar.annotate(str(value), (bar.get_width(), bar.get_y()+bar.get_height()/2),
                        xytext=(4, 0), textcoords="offset points", va="center", fontsize=8, color=TEXT)
ax_bar.set_yticks(y, TARGET_LABELS)
ax_bar.invert_yaxis()
ax_bar.set_xlabel("Positive images (count)", color=TEXT)
ax_bar.set_title("(a) Finding prevalence", loc="left", color=TEXT, weight="semibold")
ax_bar.grid(axis="x", alpha=0.25, zorder=0)
ax_bar.xaxis.set_major_locator(MaxNLocator(nbins=5, integer=True))
ax_bar.margins(x=0.16)
ax_bar.legend(frameon=False, loc="lower right", ncol=2, fontsize=8)
for spine in ax_bar.spines.values(): spine.set_visible(False)

for j, lab in enumerate(TARGET_LABELS, start=1):
    ax = fig.add_subplot(gs[0,j])
    img_id, image_path, annotation_rows = examples[lab]
    im = load_grayscale_image(image_path)
    ax.imshow(np.asarray(im), cmap="gray", vmin=0, vmax=255, interpolation="none")
    colour = ORANGE if lab == "Cardiomegaly" else TEAL
    for x1,y1,x2,y2 in boxes_for_display(annotation_rows, im.size):
        ax.add_patch(patches.Rectangle((x1,y1), x2-x1, y2-y1,
                                       facecolor="none", edgecolor=colour, linewidth=1.5))
    apply_image_style(ax, f"({chr(ord('a')+j)}) {lab}")

export_figure(fig, "vindr_prevalence_and_bbox_examples")
plt.show()


## Provenance report and suggested LaTeX replacement

After executing, cite the **actual output** in the thesis. Make sure image examples are authorized for your work and use image IDs only in private provenance logs rather than in thesis artwork.

```latex
\begin{figure}[tbp]
    \centering
    \includegraphics[width=\textwidth]{figures/anatomical_view_construction_five_panel.pdf}
    \caption{Construction of anatomy-focused image variants from a single chest radiograph. The lung- and heart-masked inputs retain the original grayscale intensity within the corresponding segmentation mask and set the remaining pixels to black. The derived inputs are not independent acquisition views.}
    \label{fig:anatomical_view_construction}
\end{figure}

\begin{figure}[tbp]
    \centering
    \includegraphics[width=\textwidth]{figures/vindr_prevalence_and_bbox_examples.pdf}
    \caption{Positive-image counts for Cardiomegaly and Pleural Effusion in the fixed VinDr-CXR validation and test cohorts (left), computed from the frozen image-level evaluation manifests. Representative test-set positive radiographs (right) show radiologist bounding-box annotations. Counts denote unique positive images, not bounding-box instances.}
    \label{fig:data_vindr_distribution}
\end{figure}
```

**Notes:** Verify box alignment visually, especially if using resized images. The cohort figure includes two target labels, not all 28 validation labels; if your thesis intends an all-label prevalence figure, revise `TARGET_LABELS` and plotting layout. The publication figure displays exact positive counts, not calculated confidence intervals. The examples are chosen deterministically unless explicit IDs are configured.


In [ ]:
print("\nOutputs written to:", OUTPUT_DIR.resolve())
print("Provenance: validation =", VINDR_VAL_LABELS_CSV)
print("Provenance: test       =", VINDR_TEST_LABELS_CSV)
print("Provenance: bbox       =", VINDR_BBOX_CSV)
print("Anatomical view image:", image_id)
for lab, (im_id, _, _) in examples.items():
    print(f"Annotated {lab} example: {im_id}")
